# Temporal baselines: Full gate (D and D+P) vs 1 fps selectors, EgoEverything

Restored from the earlier comparison notebook, with the **new policy D+P** added and the fixed-interval
baselines removed.

| method | decides from | sends a frame when |
|---|---|---|
| **Full, policy D+P** (new) | gaze + IMU, no pixels | the view settles, on each REFIXATION, **and on the first frame of every PURSUIT run** |
| Full, policy D (previous) | gaze + IMU, no pixels | the view settles, or on each REFIXATION |
| 1 fps + pixel change | grey thumbnail of every frame | it differs enough from the last sent frame |
| 1 fps + DINOv2 change | DINOv2 CLS of every frame | its cosine to the last sent frame < τ_frame |
| 1 fps: all frames | nothing | always (the ceiling) |

Reported per method: **accuracy (95% interval over videos), frames per video and per minute, visual tokens
per question, decision cost (GFLOPs/min) and VLM compute per question**, with Qwen2.5-VL-3B at 448 px.

`QUICK = False` (section 8, default) runs all 12 videos (834 questions); `True` runs the 4 test videos
(204 questions), as in the earlier table. Results go to the same resumable file as before, so D and the 1 fps
baselines are reused and only D+P is new.

## Run order
1. Clone, mount, install  2. AEA links  3. Questions  4. Held-out predictions and frame selection
5. Frames (cached on Drive)  6. Alignment and coverage  6c. Baselines (no GPU)  7. Load Qwen
8. Score  9. Results

## 1. Clone, mount, install

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('vlmeval/qwen_mcq.py'), f'vlmeval/ is not on branch {BRANCH!r}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'qwen-vl-utils', 'huggingface_hub'],
               check=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
VLM_DIR   = os.path.join(DRIVE_DIR, 'vlm', 'egoeverything')
FRAMES_TAR = os.path.join(VLM_DIR, 'frames_tar')          # one .tar per video, reused later
LOCAL_FRAMES = '/content/frames_1fps'
os.makedirs(VLM_DIR, exist_ok=True)

CKPT_FULL = os.path.join(RUNS, 'eyehead')                  # the full model's 5 fold checkpoints
missing = [i for i in range(1, 6) if not os.path.exists(os.path.join(CKPT_FULL, f'fold{i}.pt'))]
assert not missing, f'missing fold checkpoints {missing} in {CKPT_FULL} -- run the eyehead notebook first'
print('full model checkpoints: 5 folds found in', CKPT_FULL)
FEAT_CACHE = os.path.join(DRIVE_DIR, 'feat_cache.npz')     # DINOv2 CLS per frame (written by gatetrain)
print('DINOv2 feature cache:', 'found' if os.path.exists(FEAT_CACHE) else 'MISSING (needed for the DINOv2 baseline)')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 2. The AEA download-links JSON

The signed links expire after ~14 days. This cell checks the copy on Drive:

- **missing or expired** → it asks you to upload the fresh `aea_download_urls.json`, checks
  its links work, and **replaces the older copies** on Drive (`MyDrive/GazeVLM/` and, if
  present, `MyDrive/aea/`, so every notebook picks up the fresh one)
- **still valid** → nothing to do

Set `FORCE_UPLOAD = True` to replace it anyway.

In [ ]:
from vlmeval.frames import links_fresh, install_links

FORCE_UPLOAD = False
LINKS = os.path.join(DRIVE_DIR, 'aea_download_urls.json')
OLD_COPIES = [LINKS] + [p for p in ['/content/drive/MyDrive/aea/aea_download_urls.json']
                        if os.path.exists(p)]

ok, msg = links_fresh(LINKS) if os.path.exists(LINKS) else (False, 'not on Drive')
print('Drive copy:', msg)
if FORCE_UPLOAD or not ok:
    from google.colab import files
    print('upload the fresh aea_download_urls.json ...')
    up = files.upload()
    name = list(up.keys())[0]
    ok, msg = links_fresh(name)
    print('uploaded file:', msg)
    assert ok, 'the uploaded JSON does not work either -- download a fresh one from projectaria.com/datasets/aea'
    install_links(name, OLD_COPIES)
    os.remove(name)                      # the copies live on Drive; keep the repo clean
    for p in OLD_COPIES:
        print('   replaced', p)
links = json.load(open(LINKS))
print(f"using {LINKS}: {len(links['sequences'])} sequences")
print('(the eyehead notebook\'s IMU calibration, section 3, can now download its VRS too)')

## 3. EgoEverything questions on the gate's videos

In [ ]:
from vlmeval import egoev

qs_all = egoev.load_questions()
aea = qs_all[qs_all.video.str.startswith('loc')]
clips_all, test = egoev.gate_clips(FOLDS, test_only=False)
gate_videos = {c['sequence'] for c in clips_all}
test_videos = {c['sequence'] for c in clips_all if c['is_test']}

qs = aea[aea.video.isin(gate_videos)].reset_index(drop=True)
clips = [c for c in clips_all if c['sequence'] in set(qs.video)]
videos = [c['sequence'] for c in clips]
n_frames = {c['sequence']: len(c['frame_no']) for c in clips}
print(f'EgoEverything: {len(qs_all):,} questions, {len(aea):,} on AEA ({aea.video.nunique()} videos)')
print(f'on the gate videos: {len(qs):,} questions on {len(videos)} videos '
      f'({qs.video.isin(test_videos).sum()} questions on {len(set(videos) & test_videos)} test videos)\n')

per = qs.groupby('video').agg(questions=('qid', 'size'), t0_max=('t0', 'max'),
                              median_dist2gaze=('dist2gaze', 'median'))
per['video_s'] = [n_frames[v] for v in per.index]
per['split'] = ['test' if v in test_videos else 'train (held out by its fold)' for v in per.index]
display(per.sort_values('questions', ascending=False).round(1))
print('answer letters:', qs.answer.value_counts().sort_index().to_dict())

## 4. Held-out gate predictions, and which frames each condition sends

Each training video is scored by the fold model that held it out for validation; test
videos by the 5-fold ensemble. The table is cached on Drive. The quick accuracy check below
should be close to the cross-validation numbers (~0.6 four-class accuracy), not higher,
which would mean a video leaked into its own training.

In [ ]:
from eyehead.oof import oof_predictions

OOF = os.path.join(VLM_DIR, 'oof_preds_full.csv')
if os.path.exists(OOF):
    oof = pd.read_csv(OOF)
    print('loaded', OOF)
else:
    oof = oof_predictions(FOLDS, CKPT_FULL)
    oof.to_csv(OOF, index=False)
    print('wrote', OOF)

o12 = oof[oof.sequence.isin(videos)]
print(f"\nheld-out 4-class accuracy on these {len(videos)} videos: "
      f"{(o12.pred == o12.quad).mean():.3f}   by source:")
display(o12.assign(correct=o12.pred == o12.quad).groupby('source').correct.agg(['mean', 'size']).round(3))

labels = {'full': egoev.video_labels(clips, oof)}
sel_all = egoev.select_conditions(clips, labels, main='full', timeout=5, hybrid_frac=0.5, seed=0)
# keep only the policies compared here (no fixed-interval baselines)
sel = {k: sel_all[k] for k in ('D+P: full', 'D: full', 'D+P all: full', 'blind (no frames)')}
json.dump(sel, open(os.path.join(VLM_DIR, 'selection_temporal_baselines.json'), 'w'))
display(egoev.budget_table(sel, n_frames).round(2))

# per video: frames sent by D and D+P (D+P adds the first frame of every PURSUIT run)
pv_budget = pd.DataFrame({c: {v: len(sel[c][v]) for v in videos} for c in ('D: full', 'D+P: full', 'D+P all: full')})
pv_budget.insert(0, 'video_s', [n_frames[v] for v in pv_budget.index])
display(pv_budget)
for c in ('D: full', 'D+P: full', 'D+P all: full'):
    print(f"{c:<14} one frame every {pv_budget.video_s.sum() / pv_budget[c].sum():.1f} s on average")

## 5. Frames: download once, cache on Drive

The 4 test videos are already cached from the pilot; the 8 others are downloaded once
(~165 MB each).

In [ ]:
from vlmeval.frames import ensure_frames, frame_path

SIZE = 448
metas = {}
for v in videos:
    metas[v] = ensure_frames(v, links, LOCAL_FRAMES, FRAMES_TAR, size=SIZE)
    m = metas[v]
    flag = 'OK' if m['n'] >= n_frames[v] else f'!! only {m["n"]} frames, labels need {n_frames[v]}'
    print(f"   {v:<26} {m['n']:4d} frames  {flag}")

## 6. Alignment check and target coverage

**Alignment:** the frame nearest each question's keyframe, with its box. The boxes should sit
on the named objects.

**Target coverage:** distance from each question's keyframe to the nearest sent frame, per policy.

In [ ]:
import matplotlib.patches as mpatches

def frame_time(v, k):
    return metas[v]['ts'][k] if k < metas[v]['n'] else k

show = qs[qs.bbox.notna() & qs.t0.notna()].sample(min(6, len(qs)), random_state=1)
fig, ax = plt.subplots(1, len(show), figsize=(3.4 * len(show), 3.8))
for a_, q in zip(np.atleast_1d(ax), show.itertuples()):
    m = metas[q.video]
    k = int(np.argmin(np.abs(np.array(m['ts']) - q.t0)))
    img = plt.imread(frame_path(m, k))
    s = img.shape[1] / m['orig_w']
    x1, y1, x2, y2 = [b * s for b in q.bbox]
    a_.imshow(img)
    a_.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='yellow', lw=2))
    a_.set_title(f"{q.object}\nt0={q.t0:.1f}s  frame {k}", fontsize=8); a_.axis('off')
plt.tight_layout(); plt.show()

cov = egoev.coverage(qs, sel, frame_time)
ctab = cov.groupby('condition', sort=False).agg(
    questions=('qid', 'size'), median_dist_s=('dist', 'median'),
    **{f'within_{t}': (f'hit_{t}', 'mean') for t in ('1s', '2s', '5s')})
for c in ('within_1s', 'within_2s', 'within_5s'):
    ctab[c] *= 100
display(ctab.round(1))

## 6c. Baselines on the same 1 fps stream (no GPU)

Every method sees one candidate frame per second and decides per frame whether to send it.
Each decides dynamically, by its own rule:

| method | decides from | sends a frame when |
|---|---|---|
| **Full, policy D+P** (eye + IMU model) | gaze + IMU only, no pixels | the view settles, on each REFIXATION, and on the first frame of each PURSUIT run |
| Full, policy D | gaze + IMU only, no pixels | the view settles, or on each REFIXATION |
| **1 fps + DINOv2 change** | DINOv2 CLS of every frame | cosine to the **last sent** frame < τ_frame (the labels' own "frame changed" threshold) |
| **1 fps + pixel change** | 64×64 grey thumbnail of every frame | mean absolute difference to the **last sent** frame > the median consecutive-frame difference (same median rule as τ_frame) |
| **1 fps: all frames** | nothing | always (the ceiling) |

The tables below show, before any VLM run: frames sent, coverage of the questions'
moments, and what one send/skip **decision costs** in FLOPs. The DINOv2 baseline uses the
same ViT-S/14 at 98 px that made the labels. The pixel and DINOv2 baselines also need the
camera to capture and process every frame; the gaze/IMU gate does not.

A sweep of each baseline's threshold is plotted, so methods can be compared across frame budgets,
not only at their default threshold.

In [ ]:
from vlmeval import baselines as bl
from eyehead import policy as pol

TAU_F = json.load(open(os.path.join(DRIVE_DIR, 'thresholds.json')))['tau_frame']
cls_by_clip = pol.load_cls(FEAT_CACHE, clips)            # L2-normalised CLS, aligned with each clip
assert all(F is not None for F in cls_by_clip), 'feat_cache.npz does not cover these videos'
thumbs = {c['sequence']: bl.thumbnails(metas[c['sequence']], c['frame_no']) for c in clips}
PIX_THR = float(np.median(np.concatenate([bl.consecutive_pixel_diffs(t) for t in thumbs.values()])))
print(f'tau_frame (DINOv2 CLS cosine) = {TAU_F:.4f}   pixel threshold (median grey MAD) = {PIX_THR:.2f}')

ALL, PIX, DINO = '1 fps: all frames', '1 fps + pixel change', '1 fps + DINOv2 change'
def add_baselines(sel, pix_thr=PIX_THR, tau=TAU_F, names=(ALL, PIX, DINO)):
    out = {n: {} for n in names}
    for c, F in zip(clips, cls_by_clip):
        v, fn = c['sequence'], c['frame_no']
        out[names[0]][v] = [int(f) for f in fn]
        out[names[1]][v] = bl.pixel_select(fn, thumbs[v], pix_thr)
        out[names[2]][v] = bl.dino_select(fn, F, tau)
    sel.update(out)
    return out
new = add_baselines(sel)

cov = pd.concat([cov[~cov.condition.isin(new)], egoev.coverage(qs, new, frame_time)], ignore_index=True)
ctab = cov.groupby('condition', sort=False).agg(
    questions=('qid', 'size'), median_dist_s=('dist', 'median'),
    **{f'within_{t}': (f'hit_{t}', 'mean') for t in ('1s', '2s', '5s')})
for c_ in ('within_1s', 'within_2s', 'within_5s'):
    ctab[c_] *= 100

ref = metas[videos[0]]
GATE = bl.gate_decision_flops()
DEC = {'D: full': GATE, 'D+P: full': GATE, 'D+P all: full': GATE, DINO: bl.vit_flops(98),
       PIX: bl.pixel_decision_flops(ref['orig_w'], ref['orig_h']), ALL: 0}
secs = sum(n_frames.values())
cmp_rows = []
for m in ['D+P: full', 'D: full', DINO, PIX, ALL]:
    fr = sum(len(x) for x in sel[m].values())
    cmp_rows.append(dict(method=m, frames_per_video=fr / len(videos), frames_per_min=60 * fr / secs,
                         within_2s=ctab.loc[m, 'within_2s'],
                         decision_flops_per_frame=DEC.get(m, 0),
                         decision_GFLOPs_per_min=60 * DEC.get(m, 0) / 1e9))
cmp0 = pd.DataFrame(cmp_rows).set_index('method')
print('before any VLM run, on the 12 videos:')
display(cmp0.style.format({'frames_per_video': '{:.1f}', 'frames_per_min': '{:.1f}', 'within_2s': '{:.1f}',
                           'decision_flops_per_frame': '{:,.0f}', 'decision_GFLOPs_per_min': '{:.4f}'}))
print(f"one decision: gaze/IMU gate {GATE:,} FLOPs vs DINOv2 {DEC[DINO]/1e9:.2f} GFLOPs "
      f"({DEC[DINO]/GATE:,.0f}x) vs pixel {DEC[PIX]/1e6:.1f} MFLOPs")

# threshold sweeps: frames sent vs coverage
sweep = []
for q_ in (20, 30, 40, 50, 60, 70, 80, 90):
    t_ = float(np.percentile(np.concatenate([bl.consecutive_pixel_diffs(t) for t in thumbs.values()]), q_))
    s_ = add_baselines({}, pix_thr=t_, names=('_a', '_p', '_d'))['_p']
    sweep.append(('pixel change', t_, s_))
for tau_ in (0.70, 0.75, 0.80, 0.85, 0.90, 0.93, 0.95):
    s_ = add_baselines({}, tau=tau_, names=('_a', '_p', '_d'))['_d']
    sweep.append(('DINOv2 change', tau_, s_))
sweep.append(('Full (policy D)', 0, sel['D: full']))
sweep.append(('Full (policy D+P)', 0, sel['D+P: full']))
rows = []
for kind, thr, s_ in sweep:
    cv = egoev.coverage(qs, {'x': s_}, frame_time)
    rows.append(dict(kind=kind, threshold=thr, frames_per_min=60 * sum(len(x) for x in s_.values()) / secs,
                     within_2s=100 * cv['hit_2s'].mean()))
sw = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 5))
for kind, col, mk in (('pixel change', 'tab:orange', 's-'), ('DINOv2 change', 'tab:green', '^-'),
                      ('Full (policy D)', 'tab:blue', '*'), ('Full (policy D+P)', 'tab:red', '*')):
    d_ = sw[sw.kind == kind].sort_values('frames_per_min')
    ax.plot(d_.frames_per_min, d_.within_2s, mk, color=col, ms=9 if mk == '*' else 5, label=kind)
ax.set_xlabel('frames sent per minute'); ax.set_ylabel('% of questions with a frame within 2 s of their moment')
ax.set_title('coverage vs frame budget (threshold sweeps)'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## 7. Load Qwen2.5-VL-3B and smoke test

Same model and resolution as the pilot (`448*448`, about 256 visual tokens per frame).

In [ ]:
from vlmeval import qwen_mcq
import time

MODEL_ID, MAX_PIXELS = 'Qwen/Qwen2.5-VL-3B-Instruct', 448 * 448
model, proc, letter_ids, dtype = qwen_mcq.load_model(MODEL_ID)
print(MODEL_ID, dtype, f'{torch.cuda.memory_allocated()/1e9:.1f} GB on GPU')

q = qs.iloc[0]
frames = [(frame_path(metas[q.video], k), metas[q.video]['ts'][k])
          for k in sel['D+P: full'][q.video]]
t0 = time.time()
p, ntok = qwen_mcq.score(model, proc, letter_ids,
                         qwen_mcq.build_messages(q.question, q.options, frames, MAX_PIXELS))
print(f'{len(frames)} frames, {ntok} prompt tokens, {time.time()-t0:.1f} s')
print('Q:', q.question)
for L, o, pi in zip('ABCDE', q.options, p):
    print(f"   {L}. {o:<60} p={pi:.3f}{'   <- answer' if L == q.answer else ''}")
assert np.isfinite(p).all(), 'NaN logits: use an L4/A100 (bf16) or lower MAX_PIXELS'

## 8. Score Full (D+P) against the 1 fps baselines

D and the 1 fps baselines are already in the results file from the earlier run and are reused;
only **D+P** is scored now (~2.5 s per question). All frames runs last.

**All 12 videos (`QUICK = False`, 834 questions):** D+P needs ~26 min for the 630 new questions. The pixel,
DINOv2 and all-frames baselines were only run on the 4 test videos before, so on 12 videos they add
~45 min each (pixel, DINOv2) and **several hours for all frames**; stop the cell whenever you like, results are
saved after every question and section 9 reports whatever has been scored. If a prompt runs out of
GPU memory the frame size is halved for that question and recorded in `max_pixels`.

In [ ]:
# QUICK = True: only the 4 TEST videos (204 questions); False: all 12 videos (834 questions)
QUICK = False
CONDITIONS = ['D+P: full', 'D: full', DINO, PIX, ALL]   # all frames last: by far the slowest
# also available: 'D+P all: full' (D + every PURSUIT frame)
missing = [c for c in CONDITIONS if c not in sel]
assert not missing, f'unknown condition names {missing}; available: {list(sel)}'

RUN_VIDEOS = sorted(set(videos) & test_videos) if QUICK else videos
qs_run = qs[qs.video.isin(RUN_VIDEOS)].reset_index(drop=True)
print(f"{len(qs_run)} questions on {len(RUN_VIDEOS)} videos x {len(CONDITIONS)} conditions")
print(pd.DataFrame({c: {v: len(sel[c][v]) for v in RUN_VIDEOS} for c in CONDITIONS})
      .assign(video_s=[n_frames[v] for v in RUN_VIDEOS]).rename_axis("frames sent per video"))

OUT = os.path.join(VLM_DIR, f"{MODEL_ID.split('/')[-1]}_px{int(MAX_PIXELS**0.5)}_all12.jsonl")
qwen_mcq.run(model, proc, letter_ids, qs_run, sel, metas, OUT, CONDITIONS, MAX_PIXELS, tag=MODEL_ID)

## 9. Results

- **Accuracy per condition**, 95% interval from resampling the 12 videos.
- **Paired comparisons** on the same questions (`diff` = a − b in accuracy points;
  `a_only_right` / `b_only_right` = questions only one condition got right; `p_mcnemar`).
- **Accuracy by target coverage** (a frame within 2 s of the keyframe sent or not).
- **Per video**, so one video with many questions cannot drive the result unseen, and
  **test vs held-out training videos** separately.

In [ ]:
res = egoev.load_results(OUT)
bad = res[res['error'].notna()] if 'error' in res else res.iloc[:0]
if len(bad):
    print(f'{len(bad)} questions not scored (oom/nan), left out below:')
    display(bad.groupby(['condition', 'error']).size())
res = res[res['error'].isna()].copy() if 'error' in res else res.copy()
res['correct'] = res['correct'].astype(int)
res = res[res.video.isin(RUN_VIDEOS)]                # only the videos of this run (section 8)
res = res[res.condition.isin(CONDITIONS + ['D+P all: full'])]   # the file also holds older conditions
res['split'] = np.where(res.video.isin(test_videos), 'test', 'train (held out)')

acc = egoev.accuracy_table(res)
display(acc.round(2))

# the comparison table: accuracy, frames, visual tokens, and compute
vc = bl.vlm_costs(model, MAX_PIXELS)
run_secs = sum(n_frames[v] for v in RUN_VIDEOS)
crow = []
for m in [x for x in ['D+P: full', 'D: full', 'D+P all: full', DINO, PIX, ALL] if x in acc.index]:
    fr = sum(len(sel[m][v]) for v in RUN_VIDEOS)
    fq = acc.loc[m, 'frames_per_q']
    crow.append({'method': m, 'accuracy %': acc.loc[m, 'accuracy'],
                 '95% CI': f"[{acc.loc[m, 'ci_low']:.1f}, {acc.loc[m, 'ci_high']:.1f}]",
                 'frames sent / video': fr / len(RUN_VIDEOS), 'frames / min': 60 * fr / run_secs,
                 'visual tokens / question': fq * vc['tokens_per_frame'],
                 'decision GFLOPs / min': 60 * DEC.get(m, 0) / 1e9,
                 'VLM vision TFLOPs / question': fq * vc['vision_flops_per_frame'] / 1e12,
                 'VLM prefill TFLOPs / question': fq * vc['llm_prefill_flops_per_frame'] / 1e12})
cost = pd.DataFrame(crow).set_index('method')
print(f"VLM: {vc['tokens_per_frame']} visual tokens per frame at {int(MAX_PIXELS**0.5)} px; "
      f"vision encoder {vc['vision_params']/1e6:.0f}M params, LLM {vc['llm_params']/1e9:.2f}B params "
      f"(FLOPs ignore attention, so they are lower bounds)")
display(cost.style.format({'accuracy %': '{:.1f}', 'frames sent / video': '{:.1f}', 'frames / min': '{:.1f}',
                           'visual tokens / question': '{:,.0f}', 'decision GFLOPs / min': '{:.4f}',
                           'VLM vision TFLOPs / question': '{:.1f}', 'VLM prefill TFLOPs / question': '{:.1f}'}))

pairs = [('D+P: full', 'D: full'), ('D+P: full', DINO), ('D+P: full', PIX), ('D+P: full', ALL),
         ('D: full', DINO), ('D: full', PIX), ('D: full', ALL), (DINO, ALL), (PIX, ALL)]
have = set(res.condition)
pt = pd.DataFrame([egoev.paired_test(res, a, b) for a, b in pairs if a in have and b in have])
print('paired, same questions (diff = a - b, accuracy points):')
display(pt.round(3))

m = res.merge(cov[['condition', 'qid', 'hit_2s']], on=['condition', 'qid'], how='left')
by_cov = m[m.condition != 'blind (no frames)'].groupby(['condition', 'hit_2s']).correct.agg(['mean', 'size'])
by_cov['mean'] *= 100
print('accuracy (%) when a frame within 2 s of the keyframe was / was not sent:')
display(by_cov.round(1).unstack())

print('accuracy (%) per video:')
pv = res.pivot_table(index='video', columns='condition', values='correct', aggfunc='mean') * 100
pv.insert(0, 'questions', res.groupby('video').qid.nunique())
display(pv.round(1))
print('accuracy (%) on test vs held-out training videos:')
display((res.pivot_table(index='split', columns='condition', values='correct', aggfunc='mean') * 100).round(1))

fig, ax = plt.subplots(figsize=(9, 5))
order = acc.sort_values('accuracy').index
ax.barh(order, acc.loc[order, 'accuracy'],
        xerr=[acc.loc[order, 'accuracy'] - acc.loc[order, 'ci_low'],
              acc.loc[order, 'ci_high'] - acc.loc[order, 'accuracy']], color='tab:blue', alpha=.8)
ax.axvline(20, ls=':', c='k', lw=1, label='chance (20%)')
ax.set_xlabel('accuracy (%)'); ax.set_title(f'{MODEL_ID}, {len(qs)} questions, {len(videos)} videos')
ax.legend(); plt.tight_layout(); plt.show()

# accuracy against frame rate: one point per method
bt = egoev.budget_table(sel, n_frames)
pts = acc.join(bt['sends_per_min'], how='left')
fig, ax = plt.subplots(figsize=(8, 5))
for name, col in [('D+P: full', 'tab:red'), ('D: full', 'tab:blue'), ('D+P all: full', 'tab:purple'),
                  (DINO, 'tab:olive'), (PIX, 'tab:brown'), (ALL, 'black')]:
    if name in pts.index:
        r = pts.loc[name]
        ax.errorbar([r['sends_per_min']], [r['accuracy']],
                    yerr=[[r['accuracy'] - r['ci_low']], [r['ci_high'] - r['accuracy']]],
                    fmt='s', color=col, capsize=3, label=name)
ax.set_xlabel('frames sent per minute of video'); ax.set_ylabel('accuracy (%)')
ax.set_title('accuracy vs frame rate'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

---

## Reading it

| look at | meaning |
|---|---|
| D+P vs D (paired) | what sending PURSUIT frames buys, and how many extra frames it costs |
| D+P vs pixel / DINOv2 change | accuracy at (usually) fewer frames, with a decision ~30,000x cheaper than DINOv2 |
| D+P vs all frames | how much of the ceiling the gate keeps |
| accuracy vs frame rate plot | a method is better only if it is higher at the same or lower frame rate |

Caveats: 4 test videos (204 questions) by default, so intervals are wide (about ±10 points);
the baselines' frame counts differ from D+P's, so read accuracy together with frames per minute.